In [1]:
import h5py
import numpy as np
import meshio

import matplotlib.pyplot as plt

from MeshManager import MeshManager

#Set fonts
from matplotlib import rc, rcParams, cm
rc('font', **{'family': 'sans-serif', 'sans-serif': ['Arial']})

In [2]:
# Plotting function
def PlotLine(y = 2*56.25/25.4, x = 1.5*56.25/25.4, dpi=100):
    fig, ax = plt.subplots(figsize=(y, x), dpi=dpi, tight_layout=True)
    return ax

In [ ]:
# Element name
elementName = "quad"  		# meshio compatible element name
mesh = MeshManager("../SmoothAxi.msh", elementName)

In [4]:
y1 = mesh.getNodesByGroup("Upper_gauge")
y2 = mesh.getNodesByGroup("Lower_gauge")

dofyt = int(2*y1[0]+1)
dofyb = int(2*y2[0]+1)

nSteps = 9
topY_DOFs = 2*mesh.getNodesByGroup("Top")+1

force_y_e_7 = np.zeros((nSteps))
disp_y_e_7 = np.zeros((nSteps))

In [7]:
with h5py.File("SRB_5e_7.mech.out.hdf5", "a") as fh5:
    
    for iStep in range(nSteps):
        yt = fh5["Disp/Step_"+str(iStep)][()][dofyt]
        yb = fh5["Disp/Step_"+str(iStep)][()][dofyb]
        disp_y_e_7[iStep] = yt - yb
        force_y_e_7[iStep] = fh5["Force/Step_"+str(iStep)][()][topY_DOFs].sum()

In [ ]:
ax = PlotLine(dpi=200)

ax.plot(disp_y_e_7[:200]/0.03, force_y_e_7[:200]*1e-6/(np.pi*(0.003)**2), "royalblue", lw=1, ls="--", label="FEM 5x$10^{-7}$ s$^{-1}$")

ax.set_xlabel("Eng Strain")
ax.set_ylabel("Eng Stress (MPa)")

ax.legend(frameon=False, fontsize=7)

# plt.savefig("Rateline.svg", transparent=True)